# Employee Attrition Analysis - Model Training & Evaluation

## Overview
This notebook covers the full **Machine Learning pipeline**: feature encoding, preprocessing, Random Forest training with hyperparameter tuning via `RandomizedSearchCV`, model evaluation, SHAP explainability, and artifact export.

**Target**: `Attrition` (Yes / No) — Binary Classification

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
import os

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# Load clean dataset
df = pd.read_csv('../data/clean_data.csv')
print(f"Dataset loaded: {df.shape}")
df.head()

### Data Loading
We load the pre-engineered clean dataset (output of Notebook 1) containing **34 features** for **1,470 employees** and 3 newly engineered features (`SatisfactionIndex`, `PromotionRatio`, `YearsPerCompany`).

In [ ]:
# Define 15 input features for the dashboard (matching the prediction page)
input_cols = [
    'Age', 'Gender', 'Department', 'JobRole', 'MonthlyIncome',
    'BusinessTravel', 'DistanceFromHome', 'Education', 'JobSatisfaction',
    'EnvironmentSatisfaction', 'OverTime', 'YearsAtCompany',
    'YearsSinceLastPromotion', 'WorkLifeBalance', 'TrainingTimesLastYear'
]
target_col = 'Attrition'

df_model = df[input_cols + [target_col]].copy()

# Feature Engineering
df_model['PromotionRatio'] = df_model['YearsSinceLastPromotion'] / (df_model['YearsAtCompany'] + 1)
df_model['SatisfactionIndex'] = (
    df_model['EnvironmentSatisfaction'] +
    df_model['JobSatisfaction'] +
    df_model['WorkLifeBalance']) / 3.0

# X and y
X = df_model.drop(columns=[target_col])
y = df_model[target_col].map({'Yes': 1, 'No': 0})
print(f"Features: {X.shape[1]}, Samples: {len(X)}")
print(f"Class distribution: {y.value_counts().to_dict()}")

### Feature Selection & Encoding Strategy
We use the 15 dashboard input features + 2 engineered features = **17 total features**.
- **Categorical (Nominal)**: `Gender`, `Department`, `JobRole`, `BusinessTravel`, `OverTime` → One-Hot Encoding
- **Numerical / Ordinal**: All remaining columns → StandardScaler

In [ ]:
# Train-Test Split BEFORE fitting preprocessors (to prevent data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train positive rate: {y_train.mean():.2%}, Test positive rate: {y_test.mean():.2%}")

### Train-Test Split Analysis
We use stratified splitting with a 80/20 ratio to preserve the class imbalance ratio in both sets. The positive (Attrition = Yes) rate is consistently ~16% across both splits.

In [ ]:
# Build Preprocessing Pipeline
categorical_cols = ['Gender', 'Department', 'JobRole', 'BusinessTravel', 'OverTime']
numerical_cols = [
    'Age', 'MonthlyIncome', 'DistanceFromHome', 'YearsAtCompany',
    'YearsSinceLastPromotion', 'TrainingTimesLastYear', 'PromotionRatio',
    'SatisfactionIndex', 'Education', 'JobSatisfaction',
    'EnvironmentSatisfaction', 'WorkLifeBalance'
]

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

# Fit on train, transform train & test
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# Get feature names after encoding
encoded_cat = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols).tolist()
feature_names = numerical_cols + encoded_cat
print(f"Total features after encoding: {len(feature_names)}")

### Preprocessing Pipeline
We built a `ColumnTransformer` with two branches:
1. `StandardScaler` for numerical features — centers and scales to unit variance.
2. `OneHotEncoder` for categorical features — converts categories to binary columns, with `handle_unknown='ignore'` for robustness to unseen categories.
The preprocessor is **fit only on the training set** to prevent data leakage.

In [ ]:
# Random Forest + RandomizedSearchCV
param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'class_weight': ['balanced', 'balanced_subsample', None]
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20, cv=5, scoring='f1', random_state=42, n_jobs=-1
)
rf_search.fit(X_train_proc, y_train)
best_rf = rf_search.best_estimator_
print("Best params:", rf_search.best_params_)
print(f"Best CV F1: {rf_search.best_score_:.4f}")

### Random Forest + Hyperparameter Tuning Analysis
We used `RandomizedSearchCV` with **20 iterations** and **5-fold cross-validation**, optimizing for the F1 score (appropriate for imbalanced classes). `class_weight='balanced_subsample'` was selected as best, which automatically adjusts class weights in each bootstrap sample, directly addressing the 5:1 class imbalance.

In [ ]:
# Model Evaluation
y_pred = best_rf.predict(X_test_proc)
y_proba = best_rf.predict_proba(X_test_proc)[:, 1]

metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1 Score': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, y_proba)
}
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

### Evaluation Results
| Metric | Value |
|--------|-------|
| Accuracy | 80.6% |
| Precision | 41.1% |
| Recall | 48.9% |
| F1 Score | 44.7% |
| ROC-AUC | 77.4% |

**ROC-AUC of 0.77** is the most meaningful metric here. Accuracy appears high (80.6%) but is misleading due to class imbalance — a naive classifier predicting 'No' would achieve 84%. The model catches ~49% of actual leavers (Recall), which is a good starting point. Future improvements like threshold tuning or SMOTE oversampling could further boost Recall.

In [ ]:
# Save artifacts
os.makedirs('../models', exist_ok=True)
os.makedirs('../results', exist_ok=True)

joblib.dump(best_rf, '../models/best_model.pkl')
joblib.dump(preprocessor, '../models/preprocessor.pkl')

results_df = pd.DataFrame({'Metric': list(metrics.keys()), 'Value': list(metrics.values())})
results_df.to_csv('../results/model_results.csv', index=False)

feat_imp = pd.DataFrame({'Feature': feature_names, 'Importance': best_rf.feature_importances_})
feat_imp = feat_imp.sort_values('Importance', ascending=False)
feat_imp.to_csv('../results/feature_importance.csv', index=False)

print("Models and results saved successfully.")

In [ ]:
# SHAP Analysis
explainer = shap.TreeExplainer(best_rf)
shap_vals = explainer(X_test_proc)

# Select class 1 (Attrition = Yes) for binary classification
if len(shap_vals.shape) == 3:
    shap_vals = shap_vals[:, :, 1]

# Summary plot
shap.summary_plot(shap_vals, X_test_proc, feature_names=feature_names, show=True)
plt.title('SHAP Summary Plot')

### SHAP Summary Analysis
The SHAP summary plot shows that **`MonthlyIncome`, `Age`, `OverTime_Yes`, and `PromotionRatio`** are the most influential features for predicting attrition. High SHAP values (red dots pushing right) for `OverTime_Yes` show employees who work overtime are strongly pushed toward attrition prediction.

### Final Summary

### Q&A
- **Can we predict employee attrition?** Yes. The tuned Random Forest achieves a ROC-AUC of **0.774**, significantly better than a random classifier.
- **Which features matter most?** `MonthlyIncome`, `Age`, `OverTime`, and career progression metrics (`PromotionRatio`, `YearsAtCompany`) are the top drivers.

### Data Analysis Key Findings
- ROC-AUC of **0.774** on the held-out test set confirms good discriminative ability.
- Recall of **48.9%** means the model correctly identifies roughly half of all true leavers — a practical threshold for HR intervention.
- SHAP analysis confirms that low income and working overtime are the strongest risk signals.

### Insights or Next Steps
- Apply threshold tuning (e.g., lowering decision threshold from 0.5 to 0.35) to boost Recall at the cost of Precision for an HR-use-case where catching leavers is more important than false alarms.
- The model and preprocessor are exported and ready for the Streamlit dashboard deployment.